## Homework

> Note: sometimes your answer doesn't match one of
> the options exactly. That's fine.
> Select the option that's closest to your solution.
> If it's exactly in between two options, select the higher value.


### Dataset

In this homework, we continue using the fuel efficiency dataset.
Download it from <a href='https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv'>here</a>.

You can do it with wget:

```bash
wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv
```

The goal of this homework is to create a regression model for predicting the car fuel efficiency (column `'fuel_efficiency_mpg'`).



### Preparing the dataset

Preparation:

* Fill missing values with zeros.
* Do train/validation/test split with 60%/20%/20% distribution.
* Use the `train_test_split` function and set the `random_state` parameter to 1.
* Use `DictVectorizer(sparse=True)` to turn the dataframes into matrices.



In [4]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# Load the dataset
print("Loading dataset...")
df = pd.read_csv('./car_fuel_efficiency.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nMissing values:")
print(df.isnull().sum())

# ============================================================================
# PREPARING DATASET
# ============================================================================
print("\n" + "=" * 80)
print("PREPARING DATASET")
print("=" * 80)

# Fill missing values with zeros
df = df.fillna(0)

# Split the data: 60% train, 20% validation, 20% test
# First split: 60% train, 40% temp (validation + test)
df_train, df_temp = train_test_split(df, test_size=0.4, random_state=1)

# Second split: split temp into 50% validation, 50% test (which gives 20% each of original)
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=1)

print(f"Train size: {len(df_train)} ({len(df_train) / len(df) * 100:.1f}%)")
print(f"Validation size: {len(df_val)} ({len(df_val) / len(df) * 100:.1f}%)")
print(f"Test size: {len(df_test)} ({len(df_test) / len(df) * 100:.1f}%)")

# Separate features and target
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

# Remove target from features
df_train_features = df_train.drop('fuel_efficiency_mpg', axis=1)
df_val_features = df_val.drop('fuel_efficiency_mpg', axis=1)
df_test_features = df_test.drop('fuel_efficiency_mpg', axis=1)

# Convert to dictionaries for DictVectorizer
train_dicts = df_train_features.to_dict(orient='records')
val_dicts = df_val_features.to_dict(orient='records')
test_dicts = df_test_features.to_dict(orient='records')

# Use DictVectorizer with sparse=True
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

print(f"\nFeature matrix shape:")
print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")


Loading dataset...
Dataset shape: (9704, 11)

First few rows:
   engine_displacement  num_cylinders  horsepower  vehicle_weight  \
0                  170            3.0       159.0     3413.433759   
1                  130            5.0        97.0     3149.664934   
2                  170            NaN        78.0     3079.038997   
3                  220            4.0         NaN     2542.392402   
4                  210            1.0       140.0     3460.870990   

   acceleration  model_year  origin fuel_type         drivetrain  num_doors  \
0          17.7        2003  Europe  Gasoline    All-wheel drive        0.0   
1          17.8        2007     USA  Gasoline  Front-wheel drive        0.0   
2          15.1        2018  Europe  Gasoline  Front-wheel drive        0.0   
3          20.2        2009     USA    Diesel    All-wheel drive        2.0   
4          14.4        2009  Europe  Gasoline    All-wheel drive        2.0   

   fuel_efficiency_mpg  
0            13.231729 


## Question 1

Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable.

* Train a model with `max_depth=1`.


Which feature is used for splitting the data?


* `'vehicle_weight'`
* `'model_year'`
* `'origin'`
* `'fuel_type'`



In [6]:

# ============================================================================
# QUESTION 1: Decision Tree with max_depth=1
# ============================================================================
print("\n" + "=" * 80)
print("QUESTION 1: Decision Tree with max_depth=1")
print("=" * 80)

dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

# Get feature names
feature_names = dv.get_feature_names_out()

# Find the feature used for splitting
tree = dt.tree_
feature_idx = tree.feature[0]  # Root node feature
feature_name = feature_names[feature_idx]

print(f"Feature used for splitting: {feature_name}")
print(f"\nAnswer: The feature used is '{feature_name}'")



QUESTION 1: Decision Tree with max_depth=1
Feature used for splitting: vehicle_weight

Answer: The feature used is 'vehicle_weight'



## Question 2

Train a random forest regressor with these parameters:

* `n_estimators=10`
* `random_state=1`
* `n_jobs=-1` (optional - to make training faster)


What's the RMSE of this model on the validation data?

* 0.045
* 0.45
* 4.5
* 45.0


In [8]:

# ============================================================================
# QUESTION 2: Random Forest with default parameters
# ============================================================================
print("\n" + "=" * 80)
print("QUESTION 2: Random Forest RMSE")
print("=" * 80)

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_val = rf.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))

print(f"RMSE on validation data: {rmse:.6f}")
print(f"\nAnswer: {rmse:.2f}")




QUESTION 2: Random Forest RMSE
RMSE on validation data: 0.460282

Answer: 0.46



## Question 3

Now let's experiment with the `n_estimators` parameter

* Try different values of this parameter from 10 to 200 with step 10.
* Set `random_state` to `1`.
* Evaluate the model on the validation dataset.


After which value of `n_estimators` does RMSE stop improving?
Consider 3 decimal places for calculating the answer.

- 10
- 25
- 80
- 200

If it doesn't stop improving, use the latest iteration number in
your answer.


In [15]:

# ============================================================================
# QUESTION 3: n_estimators experimentation
# ============================================================================
print("\n" + "=" * 80)
print("QUESTION 3: Experimenting with n_estimators")
print("=" * 80)

n_estimators_values = range(10, 201, 10)
rmse_scores = []

print("\nn_estimators | RMSE")
print("-" * 30)

for n in n_estimators_values:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    rmse_scores.append(rmse)
    print(f"{n:12d} | {rmse:.3f}")

# Find when RMSE stops improving (considering 3 decimal places)
rmse_rounded = [round(score, 3) for score in rmse_scores]
best_rmse = min(rmse_rounded)
best_idx = rmse_rounded.index(best_rmse)

# Find the first occurrence where we reach the best RMSE
for i, rmse in enumerate(rmse_rounded):
    if rmse == best_rmse:
        best_n_estimators = list(n_estimators_values)[i]
        break

print(f"\nBest RMSE (3 decimal places): {best_rmse:.3f}")
print(f"First n_estimators to achieve this: {best_n_estimators}")
print(f"\nAnswer: {best_n_estimators}")


QUESTION 3: Experimenting with n_estimators

n_estimators | RMSE
------------------------------
          10 | 0.460
          20 | 0.446
          30 | 0.440
          40 | 0.438
          50 | 0.437
          60 | 0.436
          70 | 0.436
          80 | 0.436
          90 | 0.435
         100 | 0.435
         110 | 0.435
         120 | 0.435
         130 | 0.435
         140 | 0.435
         150 | 0.435
         160 | 0.435
         170 | 0.435
         180 | 0.435
         190 | 0.435
         200 | 0.435

Best RMSE (3 decimal places): 0.435
First n_estimators to achieve this: 90

Answer: 90


## Question 4

Let's select the best `max_depth`:

* Try different values of `max_depth`: `[10, 15, 20, 25]`
* For each of these values,
  * try different values of `n_estimators` from 10 till 200 (with step 10)
  * calculate the mean RMSE
* Fix the random seed: `random_state=1`


What's the best `max_depth`, using the mean RMSE?

* 10
* 15
* 20
* 25


In [10]:

# ============================================================================
# QUESTION 4: Best max_depth using mean RMSE
# ============================================================================
print("\n" + "=" * 80)
print("QUESTION 4: Best max_depth using mean RMSE")
print("=" * 80)

max_depth_values = [10, 15, 20, 25]
n_estimators_range = range(10, 201, 10)

results = {}

for max_depth in max_depth_values:
    rmse_list = []

    for n_est in n_estimators_range:
        rf = RandomForestRegressor(
            n_estimators=n_est,
            max_depth=max_depth,
            random_state=1,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmse_list.append(rmse)

    mean_rmse = np.mean(rmse_list)
    results[max_depth] = mean_rmse
    print(f"max_depth={max_depth}: mean RMSE = {mean_rmse:.6f}")

best_max_depth = min(results, key=results.get)
print(f"\nBest max_depth: {best_max_depth} (mean RMSE: {results[best_max_depth]:.6f})")
print(f"\nAnswer: {best_max_depth}")



QUESTION 4: Best max_depth using mean RMSE
max_depth=10: mean RMSE = 0.436247
max_depth=15: mean RMSE = 0.437825
max_depth=20: mean RMSE = 0.437693
max_depth=25: mean RMSE = 0.437653

Best max_depth: 10 (mean RMSE: 0.436247)

Answer: 10




# Question 5

We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split.
When doing it, we can calculate "gain" - the reduction in impurity before and after the split.
This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the
[`feature_importances_`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor.feature_importances_)
field.

For this homework question, we'll find the most important feature:

* Train the model with these parameters:
  * `n_estimators=10`,
  * `max_depth=20`,
  * `random_state=1`,
  * `n_jobs=-1` (optional)
* Get the feature importance information from this model


What's the most important feature (among these 4)?

* `vehicle_weight`
*	`horsepower`
* `acceleration`
* `engine_displacement`



In [14]:

# ============================================================================
# QUESTION 5: Feature Importance
# ============================================================================
print("\n" + "=" * 80)
print("QUESTION 5: Most Important Feature")
print("=" * 80)

rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Get feature importances
feature_importances = rf.feature_importances_
feature_names = dv.get_feature_names_out()

# Create a dataframe for better visualization
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(importance_df.head(10).to_string(index=False))

# Find the most important among the specified features
target_features = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
target_importances = {}

for feat in target_features:
    importance = importance_df[importance_df['feature'] == feat]['importance'].values
    if len(importance) > 0:
        target_importances[feat] = importance[0]
    else:
        target_importances[feat] = 0

print(f"\nImportance of target features:")
for feat, imp in sorted(target_importances.items(), key=lambda x: x[1], reverse=True):
    print(f"  {feat}: {imp:.6f}")

most_important = max(target_importances, key=target_importances.get)
print(f"\nMost important feature: {most_important}")
print(f"\nAnswer: {most_important}")



QUESTION 5: Most Important Feature

Top 10 Most Important Features:
            feature  importance
     vehicle_weight    0.959878
         horsepower    0.015933
       acceleration    0.011442
engine_displacement    0.003159
         model_year    0.003066
      num_cylinders    0.002323
          num_doors    0.001576
         origin=USA    0.000496
        origin=Asia    0.000431
      origin=Europe    0.000419

Importance of target features:
  vehicle_weight: 0.959878
  horsepower: 0.015933
  acceleration: 0.011442
  engine_displacement: 0.003159

Most important feature: vehicle_weight



## Question 6

Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:

* Install XGBoost
* Create DMatrix for train and validation
* Create a watchlist
* Train a model with these parameters for 100 rounds:

```
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}
```

Now change `eta` from `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?

* 0.3
* 0.1
* Both give equal value


In [13]:

# ============================================================================
# QUESTION 6: XGBoost eta parameter tuning
# ============================================================================
print("\n" + "=" * 80)
print("QUESTION 6: XGBoost eta parameter tuning")
print("=" * 80)

import xgboost as xgb

# Create DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Create watchlist
watchlist = [(dtrain, 'train'), (dval, 'validation')]

# Test eta = 0.3
print("\nTraining with eta=0.3...")
xgb_params_03 = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

model_03 = xgb.train(
    xgb_params_03,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=False
)

y_pred_03 = model_03.predict(dval)
rmse_03 = np.sqrt(mean_squared_error(y_val, y_pred_03))

# Test eta = 0.1
print("Training with eta=0.1...")
xgb_params_01 = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

model_01 = xgb.train(
    xgb_params_01,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=False
)

y_pred_01 = model_01.predict(dval)
rmse_01 = np.sqrt(mean_squared_error(y_val, y_pred_01))

print(f"\nResults:")
print(f"  eta=0.3: RMSE = {rmse_03:.6f}")
print(f"  eta=0.1: RMSE = {rmse_01:.6f}")

if rmse_03 < rmse_01:
    print(f"\nAnswer: 0.3 (better RMSE)")
elif rmse_01 < rmse_03:
    print(f"\nAnswer: 0.1 (better RMSE)")
else:
    print(f"\nAnswer: Both give equal value")





QUESTION 6: XGBoost eta parameter tuning

Training with eta=0.3...
Training with eta=0.1...

Results:
  eta=0.3: RMSE = 0.443405
  eta=0.1: RMSE = 0.416743

Answer: 0.1 (better RMSE)



## Submit the results

* Submit your results here: https://courses.datatalks.club/ml-zoomcamp-2025/homework/hw06
* If your answer doesn't match options exactly, select the closest one. If the answer is exactly in between two options, select the higher value.
